# Exercise 4.1 — One workflow, two classifiers

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day4/day4-01-frameworks.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

In Exercise 3.1 you predicted a number, blood glucose. Now the outcome is a
**class**: does this person have diabetes, `pos` or `neg`? The steps are
exactly the same: **specify** a model, **fit** it on the training patients, and
**predict** for new ones. Two things are new:

- the specification says the model is for **classification**;
- the prediction can be a **probability** for each class, as well as the class
  itself.

|  | Exercise 3.1: glucose, a number | Exercise 4.1: diabetes, a class |
|---|---|---|
| specify (R) | `linear_reg()`, `decision_tree()` with `set_mode("regression")` | `logistic_reg()`, `decision_tree()` with `set_mode("classification")` |
| specify (Python) | `LinearRegression()`, `DecisionTreeRegressor()` | `LogisticRegression()`, `DecisionTreeClassifier()` |
| fit | `fit()` / `.fit()` | the same |
| predict | a number | a class, or a probability for each class |

- **Part 1.** Predict diabetes from glucose with a logistic regression. Get the
  probability and the class for one patient, see the patient on the fitted
  curve, and compare the predictions with what was measured in the test
  patients.
- **Part 2.** Add BMI and predict for the same patient again. See the
  probability as a map over glucose and BMI.
- **Part 3.** Keep the same steps, training patients and predictors, and fit a
  decision tree instead. Follow the patient to a leaf, read the probability and
  the class there, and compare the tree's map with the logistic regression's.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree

`data/diabetes_clean.csv` holds 750 adults screened for type 2 diabetes: eight
measurements from a single visit, and `diabetes`, which is `pos` or `neg`. It
is ready to use. We keep glucose, BMI (the column `mass`) and the outcome. If
you do not have it yet, [**download `diabetes_clean.csv`**](data/diabetes_clean.csv)
into your `data` folder.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["glucose", "mass", "diabetes"]]

diabetes.describe()

We hold back every fifth person as the test set: 600 training patients and 150
test patients. Splitting by a fixed rule rather than at random gives both
languages the same 150 people, so this tab prints the same numbers as the R
tab. Every Day 4 exercise uses this split.

In [ ]:
is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

train["diabetes"].value_counts()

### Part 1 — Predict diabetes from glucose

The same three steps as in Exercise 3.1. The class name says that this model
is for classification: `LogisticRegression` here, where Day 3 had
`LinearRegression`.

In [ ]:
# 1. Specify: logistic regression
log_model = LogisticRegression(C=np.inf)

# 2. Fit on the training patients: predictors (a table) and outcome
log_model.fit(train[["glucose"]], train["diabetes"])

# the classes, and the learned parameters
print("classes:", log_model.classes_)
print("intercept:", log_model.intercept_)
print("weight for glucose:", log_model.coef_)

`C=np.inf` switches off a penalty that scikit-learn adds to logistic regression
by default; without it the parameters would not match the ones in the R tab.

The two parameters are not on the glucose scale but on the **log-odds** scale.
Exercise 4.2 shows how they turn into a probability. Here we let scikit-learn
do it.

#### Task 1a — A patient with a glucose of 150

`.predict()` gives the **class**. `.predict_proba()` gives a **probability** for
each class instead, one column per class in the order of `classes_`: `neg`
first, then `pos`.

In [ ]:
# Replace every ______ below, then run this cell.
patient = pd.DataFrame({"glucose": [150]})

print(log_model.______(patient))   # a probability for each class: neg, pos
print(log_model.predict(patient))  # the class

> **Stuck?**
>
> The method's name is `predict` followed by an underscore and a short word for
> probability.

The class is the probability cut at 0.5: `pos` if $\hat{P}(\text{pos}) > 0.5$,
`neg` otherwise. The red point is the patient, on the fitted curve:

In [ ]:
curve = pd.DataFrame({"glucose": np.arange(40, 201, 1)})
curve_prob = log_model.predict_proba(curve)[:, 1]
jitter = np.random.default_rng(1).uniform(-0.03, 0.03, len(train))

fig, ax = plt.subplots()
ax.scatter(train["glucose"], (train["diabetes"] == "pos") + jitter,
           alpha=0.3, color="steelblue")
ax.plot(curve["glucose"], curve_prob, color="red", linewidth=2)
ax.axhline(0.5, linestyle="--", color="grey")
ax.scatter([150], log_model.predict_proba(patient)[:, 1], color="red", s=80, zorder=3)
ax.set_xlabel("Glucose (mg/dL)")
ax.set_ylabel("Probability of diabetes")
ax.set_title("Training patients (0 = neg, 1 = pos), the fitted curve, and the patient")
plt.show()

> **Think about it**
>
> 1. The model says `pos` for this patient. How sure is it?
> 2. At about what glucose does the predicted class change from `neg` to `pos`?

#### Task 1b — Predictions against what was measured

The test patients were not used to fit the model. Predict both the probability
of `pos` and the class for each of them, and put them next to what was
measured.

In [ ]:
test_results = test.copy()
test_results["prob_pos"] = log_model.predict_proba(test[["glucose"]])[:, 1]
test_results["predicted"] = log_model.predict(test[["glucose"]])

test_results.head(10)

How often is the predicted class right? Compare it with a "model" that always
says `neg`, the most common class.

In [ ]:
# Replace every ______ below, then run this cell.
print((test_results["predicted"] == test_results["______"]).mean())   # accuracy of the model
print((test_results["diabetes"] == "neg").mean())                    # always saying neg

> **Stuck?**
>
> Compare the predicted class with the column that holds what was measured.

In [ ]:
fig, ax = plt.subplots()
ax.axhline(0.5, linestyle="--", color="grey")
for cls, colour in [("neg", "steelblue"), ("pos", "darkorange")]:
    rows = test_results[test_results["diabetes"] == cls]
    ax.scatter(rows["glucose"], rows["prob_pos"], color=colour, alpha=0.7, label=cls)
ax.set_xlabel("Glucose (mg/dL)")
ax.set_ylabel("Predicted probability of diabetes")
ax.legend(title="Measured")
ax.set_title("Test patients: predicted probability, coloured by what was measured")
plt.show()

> **Think about it**
>
> 1. Every point lies on the same curve. What has the model got wrong for the
>    orange points below the dashed line, and for the blue points above it?
> 2. Which is more useful to a clinician: the class, or the probability?

### Part 2 — Add BMI

The same kind of model, with a second predictor:

In [ ]:
log_model_2 = LogisticRegression(C=np.inf)
log_model_2.fit(train[["glucose", "mass"]], train["diabetes"])

print("intercept:", log_model_2.intercept_)
print("weights (glucose, mass):", log_model_2.coef_)

#### Task 2a — The same patient, now with a BMI of 30

In [ ]:
# Replace every ______ below, then run this cell.
patient = pd.DataFrame({"glucose": [150], "mass": [______]})

print(log_model_2.______(patient))
print(log_model_2.predict(patient))

> **Stuck?**
>
> The patient now needs a value for both predictors. Ask for the probabilities
> exactly as in Task 1a.

With two predictors, the probability is a **map** over glucose and BMI. The
black line is where the probability is exactly 0.5: to its right the class is
`pos`, to its left `neg`. Circles are test patients without diabetes, triangles
test patients with it, and the red point is the patient.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# every combination of glucose and BMI on a fine grid
G, M = np.meshgrid(np.arange(40, 201, 1), np.arange(18, 68.5, 0.5))
grid = pd.DataFrame({"glucose": G.ravel(), "mass": M.ravel()})
blue_orange = LinearSegmentedColormap.from_list("bo", ["steelblue", "white", "darkorange"])

def prob_map(model, title):
    prob = model.predict_proba(grid)[:, 1].reshape(G.shape)
    fig, ax = plt.subplots()
    im = ax.pcolormesh(G, M, prob, cmap=blue_orange, vmin=0, vmax=1, shading="auto")
    ax.contour(G, M, prob, levels=[0.5], colors="black")
    neg, pos = test[test["diabetes"] == "neg"], test[test["diabetes"] == "pos"]
    ax.scatter(neg["glucose"], neg["mass"], marker="o", facecolors="none",
               edgecolors="black", s=20, label="test: neg")
    ax.scatter(pos["glucose"], pos["mass"], marker="^", color="black", s=20,
               label="test: pos")
    ax.scatter([150], [30], color="red", s=80, zorder=3, label="patient")
    fig.colorbar(im, ax=ax, label="Probability of diabetes")
    ax.set_xlabel("Glucose (mg/dL)")
    ax.set_ylabel("BMI (kg/m²)")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    plt.show()

prob_map(log_model_2, "Logistic regression: glucose and BMI")

> **Think about it**
>
> 1. The line where the class changes is straight but tilted. What does the tilt
>    tell you about BMI?
> 2. Adding BMI moved this patient from 0.547 to 0.490, and the class from `pos`
>    to `neg`. Did BMI change much about this patient's prediction?

### Part 3 — The same steps, a decision tree

Only the specification changes. The training patients, the predictors and the
`.fit()`, `.predict()` and `.predict_proba()` calls stay the same.

In [ ]:
# 1. Specify: a classification tree, kept small
tree_model = DecisionTreeClassifier(max_depth=2, random_state=0)

# 2. Fit on the same training patients, with the same predictors
tree_model.fit(train[["glucose", "mass"]], train["diabetes"])

fig, ax = plt.subplots(figsize=(10, 5))
plot_tree(tree_model, feature_names=["glucose", "mass"], class_names=["neg", "pos"],
          filled=False, impurity=False, proportion=True, precision=2, ax=ax)
plt.show()

`max_depth=2` allows at most two questions in a row. Without it the tree keeps
splitting until every leaf holds patients of one class only: 205 leaves, which
fit the training patients almost one by one and are right for only 64% of the
test patients.

Each box asks a yes/no question. **True goes left, False goes right.** In each
leaf at the bottom, `samples` is the percentage of all training patients who
ended up there, `value` gives the share of them in each class (`neg`, `pos`),
and `class` is the predicted class. The share of `pos` is the leaf's
probability of diabetes.

#### Task 3a — Follow the patient

Start at the top with the patient with a glucose of 150 and a BMI of 30. At
each question, go left or right. Which leaf do you reach? What probability and
what class does it give?

Check it with `.predict_proba()` and `.predict()`:

In [ ]:
print(tree_model.predict_proba(patient))
print(tree_model.predict(patient))

> **Why the R tab's tree is smaller**
>
> The R tab fits the same training patients and gets three leaves, not four: it
> does not ask about BMI for patients with a glucose up to 154.5. `rpart` keeps a
> question only if it changes the predicted class of some patients, and here both
> answers still say `neg`. scikit-learn keeps any question that makes the leaves
> purer: it separates patients with a low BMI (7% have diabetes) from the rest
> (31%). The probabilities change; the classes do not.

The same map as for the logistic regression:

In [ ]:
prob_map(tree_model, "Decision tree: glucose and BMI")

How often is each model right on the test patients?

In [ ]:
pd.DataFrame({
    "model": ["logistic, glucose", "logistic, glucose + BMI", "tree, glucose + BMI",
              "always says neg"],
    "accuracy": [log_model.score(test[["glucose"]], test["diabetes"]),
                 log_model_2.score(test[["glucose", "mass"]], test["diabetes"]),
                 tree_model.score(test[["glucose", "mass"]], test["diabetes"]),
                 (test["diabetes"] == "neg").mean()]})

`.score()` is the accuracy: the share of patients whose predicted class is
right.

> **Think about it**
>
> 1. The logistic regression gives every patient their own probability. How many
>    different probabilities can the tree give, and why?
> 2. Compare the two maps. What shape is the region where each model says `pos`?
> 3. The three models gave the patient 0.55, 0.49 and 0.31. Their accuracies on
>    the test patients are close. What does that tell you?

> **Learning points**
>
> - The steps are the same as in Exercise 3.1: specify, fit on the training
>   patients, predict for new ones. For a class instead of a number, only the
>   estimator changes: `LogisticRegression` and `DecisionTreeClassifier`.
> - `.predict()` gives the class; `.predict_proba()` gives a probability for each
>   class. The class is the probability cut at 0.5.
> - The probability carries more information than the class: near 0.5 a small
>   change flips the class.
> - A logistic regression's probability changes smoothly and its classes are
>   split by a straight line; a tree gives one probability per leaf and splits
>   the classes into rectangles.